<a href="https://colab.research.google.com/github/gibranfp/CursoAprendizajeProfundo/blob/2026-1/notebooks/3b_traduccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traducción automática en redes neuronales recurrentes
En esta libreta entrenaremos y evaluaremos un traductor automático basados en redes neuronales recurrentes.

In [1]:
import copy
import requests
import gzip

import numpy as np
import matplotlib.pyplot as plt

import spacy
from collections import Counter

import torch as th
from torch import nn

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

DEVICE = 'cuda:0' if th.cuda.is_available() else 'cpu'
EMBED_DIM = 128
HIDDEN_DIM = 128
N_LAYERS = 3
P_DROPOUT = 0.6

BATCH_SIZE = 64
N_EPOCAS = 20

np.random.seed(42)
th.random.manual_seed(42)

## Conjunto de datos Multi30k
Definimos funciones para descargar los archivos `.gz` del conjunto de datos Multi30k a partir de la URL:

In [2]:
def download_from_url(url, fname):
  with open(fname, 'wb') as f:
    res = requests.get(url)
    res.raise_for_status()
    f.write(res.content)

Descargamos los archivos de entrenamiento, validación y prueba para los idiomas fuente y objetivo:

In [3]:
url_base = 'https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/'
train_urls = ('train.fr.gz', 'train.en.gz')
val_urls = ('val.fr.gz', 'val.en.gz')
test_urls = ('test_2016_flickr.fr.gz', 'test_2016_flickr.en.gz')

download_from_url(url_base + train_urls[0], train_urls[0])
download_from_url(url_base + train_urls[1], train_urls[1])
download_from_url(url_base + val_urls[0], val_urls[0])
download_from_url(url_base + val_urls[1], val_urls[1])

Cargamos los analizadores léxicos (_tokenizadores_) de la biblioteca [spaCy](https://spacy.io/) para el idioma fuente y el idioma objetivo:

In [4]:
!python -m spacy download fr_core_news_sm
!python -m spacy download en_core_web_sm

tok_fr = spacy.load("fr_core_news_sm")
tok_en = spacy.load("en_core_web_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 52.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 39.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


El conjunto de datos consiste de archivos comprimidos de texto con una línea por oración. Las líneas del archivo del idioma fuente y del idioma objetivo están emparejados (es decir, una línea específica contiene la misma oración en cada idioma).

Leemos la primera línea de los archivos de los idiomas fuente y objetivo y las pasamos por el analizador léxico:

In [5]:
with gzip.open(train_urls[0], 'rt', encoding='utf-8') as f:
  txt_fr = f.readline()

with gzip.open(train_urls[1], 'rt', encoding='utf-8') as f:
  txt_en = f.readline()

print(f'Francés: {txt_fr} --> {[t.text for t in tok_fr(txt_fr)]}')
print(f'Inglés: {txt_en} --> {[t.text for t in tok_en(txt_en)]}')

Francés: Deux jeunes hommes blancs sont dehors près de buissons.
 --> ['Deux', 'jeunes', 'hommes', 'blancs', 'sont', 'dehors', 'près', 'de', 'buissons', '.', '\n']
Inglés: Two young, White males are outside near many bushes.
 --> ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '\n']


Definimos una función para cargar todas las líneas de un archivo en memoria:

In [6]:
def read_sentences(filepath):
  with gzip.open(filepath, 'rt', encoding='utf-8') as f:
    return f.readlines()

Cargamos las líneas procesadas por el analizador léxico de los archivos de los idiomas fuente y objetivo:

In [7]:
sentences_fr = [[t for t in tok_fr(l)] for l in read_sentences(train_urls[0])]
sentences_en = [[t for t in tok_en(l)] for l in read_sentences(train_urls[1])]

Definimos una clase y función para construir el vocabulario.

In [8]:
def build_vocab(sentences):
  return Counter([t.text for s in sentences for t in s])

class VocabFrEn:
  def __init__(self,
               sentences_fr,
               sentences_en,
               specials = ['<PAD>', '<CLS>', '<SEP>', '<OOV>']):
    offset = len(specials)
    vocab_fr = build_vocab(sentences_fr)
    vocab_en = build_vocab(sentences_en)

    self.id2str_fr = {i+offset:s for i,(s,f) in enumerate(vocab_fr.most_common())}
    self.str2id_fr = {s:i+offset for i,(s,f) in enumerate(vocab_fr.most_common())}

    self.id2str_en = {i+offset:s for i,(s,f) in enumerate(vocab_en.most_common())}
    self.str2id_en = {s:i+offset for i,(s,f) in enumerate(vocab_en.most_common())}

    for i,s in enumerate(specials):
      self.str2id_fr[s] = i
      self.id2str_fr[i] = s
      self.str2id_en[s] = i
      self.id2str_en[i] = s

Instanciamos el vocabulario y agregamos los _tokens_ especiales `<CLS>` y `<SEP>`.

In [9]:
vocab = VocabFrEn(sentences_fr, sentences_en)
SRC_PAD_IDX = vocab.str2id_fr['<CLS>']
TGT_PAD_IDX = vocab.str2id_en['<SEP>']

Creamos la clase que define el iterable del conjunto de datos Multi30k.

In [10]:
class Multi30k(Dataset):
  def __init__(self,
               path_fr,
               path_en,
               tok_fr,
               tok_en,
               vocab):
    self.tok_fr = tok_fr
    self.tok_en = tok_en
    self.vocab = vocab

    self.sentences_fr = []
    self.sentences_en = []

    clsid_fr = vocab.str2id_fr['<CLS>']
    sepid_fr = vocab.str2id_fr['<SEP>']
    oovid_fr = vocab.str2id_fr['<OOV>']
    clsid_en = vocab.str2id_en['<CLS>']
    sepid_en = vocab.str2id_en['<SEP>']
    oovid_en = vocab.str2id_en['<OOV>']
    for sfr,sen in zip(read_sentences(path_fr), read_sentences(path_en)):
      sent_fr = [vocab.str2id_fr[t.text] if t.text in vocab.str2id_fr else oovid_fr
                 for t in tok_fr(sfr)[:-1]]
      sent_fr.insert(0, clsid_fr)
      sent_fr.append(sepid_fr)
      self.sentences_fr.append(th.tensor(sent_fr))

      sent_en = [vocab.str2id_en[t.text] if t.text in vocab.str2id_en else oovid_en
                 for t in tok_en(sen)[:-1]]
      sent_en.insert(0, clsid_en)
      sent_en.append(sepid_en)
      self.sentences_en.append(th.tensor(sent_en))

  def __getitem__(self, idx):
    return self.sentences_fr[idx], self.sentences_en[idx]

  def __len__(self):
    return len(self.sentences_fr)

Instanciamos la clase para los subconjuntos de entrenamiento y validación.

In [11]:
entds = Multi30k(train_urls[0], train_urls[1], tok_fr, tok_en, vocab)
valds = Multi30k(val_urls[0], val_urls[1], tok_fr, tok_en, vocab)

Tomamos un elemento del iterable del subconjunto de entrenamiento.

In [12]:
it = iter(entds)
src, tgt = next(it)
src, tgt

(tensor([   1,   26,   85,   34,  225,   31,   91,   75,    9, 1193,    5,    2]),
 tensor([   1,   20,   26,   16, 1166,  805,   18,   58,   85,  335, 1330,    6,
            2]))

Definimos una función para convertir secuencias de índices a cadenas de caracteres.

In [13]:
def convert_indexes(ten, id2str):
  return [id2str[idx.item()] for idx in ten]

print(convert_indexes(src, vocab.id2str_fr))
print(convert_indexes(tgt, vocab.id2str_en))

['<CLS>', 'Deux', 'jeunes', 'hommes', 'blancs', 'sont', 'dehors', 'près', 'de', 'buissons', '.', '<SEP>']
['<CLS>', 'Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '<SEP>']


Definimos una clase para agregar _padding_ a las secuencias y para crear las secuencias del idioma objetivo para idioma de salida deseada y para el _teacher forcing_.  

In [14]:
class PadSequences:
  def __call__(self,
               batch):
    # Quitamos \n a secuencia de idioma fuente
    src = [item[0][:-1] for item in batch]
    src = pad_sequence(src,
                       batch_first = False,
                       padding_value = SRC_PAD_IDX)

    # Secuencia de idioma objetivo que se usará como salida deseada
    # quitando \n
    tgt_out = [item[1][1:] for item in batch]
    tgt_out = pad_sequence(tgt_out,
                           batch_first = False,
                           padding_value = TGT_PAD_IDX)

    # Secuencia de idioma objetivo que se usará como entrada para el teacher forcing
    # quitando \n
    tgt_inp = [item[1][:-1] for item in batch]
    tgt_inp = pad_sequence(tgt_inp,
                           batch_first = False,
                           padding_value = TGT_PAD_IDX)

    return src, tgt_inp, tgt_out

Formamos los cargadores de datos.

In [15]:
entdl = DataLoader(entds,
                   batch_size = BATCH_SIZE,
                   shuffle = True,
                   drop_last = True,
                   collate_fn = PadSequences())

valdl = DataLoader(valds,
                   batch_size = BATCH_SIZE,
                   shuffle = False,
                   drop_last = False,
                   collate_fn = PadSequences())

itdl = iter(entdl)
src_batch, tgt_inp_batch, tgt_out_batch = next(itdl)
src_batch = src_batch.to(DEVICE)
tgt_inp_batch = tgt_inp_batch.to(DEVICE)
tgt_out_batch = tgt_out_batch.to(DEVICE)

## Arquitectura
La red neuronal tiene una arquitectura codificador-decodificador y está basada en celdas LSTM (_Long Short-Term Memory_).

### Codificador
El codificador espera como entrada una secuencia de índices $(x_c^{[1]}, \ldots, x_c^{[T]})$ que representa una oración en el idioma fuente, la cual convierte  a una secuencia de vectores densos $(\mathbf{e}_c^{[1]}, \ldots, \mathbf{e}_c^{[T]})$ usando una matriz entrenable (_embedding_). Dicha secuencia se procesa por una o varias celdas LSTM, obteniendo como resultado una secuencia de vectores de estado $(\mathbf{h}_c^{[1]}, \ldots, \mathbf{h}_c^{[T]})$, así como los últimos estados $\mathbf{h}_c^{[T]}$ y $\mathbf{C}_c^{[T]}$. Aquí, $x_c^{[i]} \in \mathbb{N}$, $\mathbf{e}_c^{[i]} \in \mathbb{R}^{D_{cod}}$ y $\mathbf{h}_c^{[i]} \in \mathbb{R}^{L_{cod}}$.

In [16]:
class Encoder(nn.Module):
  def __init__(self,
               vocab_dim,
               embed_dim,
               hidden_dim,
               n_layers,
               p_dropout = 0):
    super(Encoder, self).__init__()

    self.hidden_dim = hidden_dim
    self.n_layers = n_layers

    self.embed = nn.Embedding(vocab_dim, embed_dim)
    self.cells = nn.LSTM(embed_dim,
                         hidden_dim,
                         n_layers,
                         dropout = p_dropout)

  def forward(self, x):
    # [n_steps, n_examples] -> [n_steps, n_examples, embed_dim]
    embs = self.embed(x)

    # output: [n_steps, n_examples, hidden_dim]
    # h,c: [n_layers, n_examples, hidden_dim]
    output, (h, c) = self.cells(embs)

    return output, (h, c)

enc = Encoder(len(vocab.str2id_fr),
              EMBED_DIM,
              HIDDEN_DIM,
              N_LAYERS,
              P_DROPOUT)
enc.to(DEVICE)
output, (h, c) = enc(src_batch)
output, (h, c)

(tensor([[[ 2.0132e-02, -8.1751e-03, -1.0526e-02,  ..., -1.9578e-03,
           -1.2095e-02, -1.1773e-04],
          [ 1.5957e-02, -1.9454e-02,  3.5494e-03,  ..., -9.3162e-03,
           -2.0432e-02,  1.5174e-03],
          [ 2.2169e-02, -1.3559e-02, -2.6584e-03,  ..., -1.8243e-02,
           -1.7088e-02, -1.4349e-04],
          ...,
          [ 1.3868e-02, -1.7589e-02, -5.0552e-03,  ..., -6.8187e-03,
           -1.7005e-02,  6.3437e-04],
          [ 2.7733e-02, -3.2457e-03,  1.0069e-03,  ..., -1.2678e-02,
           -1.6785e-02,  1.7668e-03],
          [ 1.2939e-02, -6.3416e-03,  5.3374e-03,  ..., -3.5903e-03,
           -1.8066e-02, -6.3036e-03]],
 
         [[ 3.3753e-02, -2.0145e-02,  2.0849e-03,  ..., -8.9936e-03,
           -1.6099e-02, -2.6029e-03],
          [ 3.3665e-02, -1.7295e-02,  2.1216e-02,  ..., -1.2226e-02,
           -1.6002e-02,  7.8264e-03],
          [ 3.7123e-02, -2.3808e-02,  1.9955e-02,  ..., -2.4232e-02,
           -3.1606e-02, -1.5185e-02],
          ...,
    

### Codificador
El decodificador recibe una secuencia de índices $(x_d^{[1]}, \ldots, x_d^{[T]})$, la cual convierte a una secuencia de vectores densos $(\mathbf{e}_d^{[1]}, \ldots, \mathbf{e}_d^{[T]})$ usando una matriz entrenable (_embedding_). Esta secuencia de vectores densos se procesa por una o varias celdas LSTM, obteniendo como resultado una secuencia de vectores de estado $(\mathbf{h}_d^{[1]}, \ldots, \mathbf{h}_d^{[T]})$, así como los últimos estados $\mathbf{h}_d^{[T]}$ y $\mathbf{C}^{[T]}$. Aquí, $x_d^{[i]} \in \mathbb{N}$, $\mathbf{e}_d^{[i]} \in \mathbb{R}^{D_{dec}}$ y $\mathbf{h}_d^{[i]} \in \mathbb{R}^{L_{dec}}$.

In [17]:
class Decoder(nn.Module):
  def __init__(self,
               vocab_dim,
               embed_dim,
               hidden_dim,
               n_layers,
               p_dropout = 0):
    super(Decoder, self).__init__()

    self.hidden_dim = hidden_dim
    self.n_layers = n_layers
    self.vocab_dim = vocab_dim

    self.embed = nn.Embedding(vocab_dim, embed_dim)
    self.cells = nn.LSTM(embed_dim,
                         hidden_dim,
                         n_layers,
                         dropout = p_dropout)
    self.fc = nn.Linear(hidden_dim, vocab_dim)

  def forward(self, x, h0, c0):
    # [n_steps, n_examples] -> [n_steps, n_examples, embed_dim]
    embs = self.embed(x)

    # output: [n_steps, n_examples, hidden_dim]
    # h y c: [n_layers, n_examples, hidden_dim]
    output, (h, c) = self.cells(embs, (h0, c0))

    # [n_steps, n_examples, hidden_dim] -> [n_examples, vocab_dim]
    logits = self.fc(output.squeeze(0))

    return logits, (h, c)

dec = Decoder(len(vocab.str2id_en),
              EMBED_DIM,
              HIDDEN_DIM,
              N_LAYERS,
              P_DROPOUT)
dec.to(DEVICE)
l, (h, c) = dec(tgt_inp_batch, h, c)

### Secuencia a secuencia
El traductor tiene una arquitectura secuencia a secuencia que recibe dos secuencias de índices: una del idioma fuente y otra del idioma objetivo. donde un codificador procesa una secuencia de índices en el idioma fuente

In [18]:
class Seq2Seq(nn.Module):
  def __init__(self,
               vocab,
               embed_dim,
               hidden_dim,
               n_layers,
               p_dropout = 0):
    super(Seq2Seq, self).__init__()

    self.vocab = vocab
    self.src_vocab_dim = len(vocab.str2id_fr)
    self.tgt_vocab_dim = len(vocab.str2id_en)

    self.emb_dim = embed_dim
    self.hidden_dim = hidden_dim
    self.n_layers = n_layers

    self.enc = Encoder(self.src_vocab_dim,
                       embed_dim,
                       hidden_dim,
                       n_layers,
                       p_dropout)
    self.dec = Decoder(self.tgt_vocab_dim,
                       embed_dim,
                       hidden_dim,
                       n_layers,
                       p_dropout)

  def forward(self, src, tgt):
    output, (he, ce) = self.enc(src)

    logits = th.zeros(tgt.shape[1],
                      tgt.shape[0],
                      self.tgt_vocab_dim, device = tgt.device)
    logits[:, 0, :], (hd, cd) = self.dec(tgt[0:1], he, ce)
    for i in range(1, tgt.shape[0]):
      logits[:, i, :], (hd, cd) = self.dec(tgt[i:i+1], hd, cd)

    return logits

modelo = Seq2Seq(vocab,
                 EMBED_DIM,
                 HIDDEN_DIM,
                 N_LAYERS,
                 P_DROPOUT)
modelo.to(DEVICE)
modelo(src_batch, tgt_inp_batch)

tensor([[[-0.0678, -0.0277, -0.0879,  ..., -0.0201,  0.0142,  0.0788],
         [-0.0612, -0.0109, -0.0684,  ..., -0.0088,  0.0114,  0.0916],
         [-0.0596, -0.0031, -0.0544,  ..., -0.0091,  0.0146,  0.0943],
         ...,
         [-0.0595, -0.0121, -0.0706,  ..., -0.0304,  0.0133,  0.1200],
         [-0.0622, -0.0017, -0.0662,  ..., -0.0325,  0.0098,  0.1138],
         [-0.0655, -0.0019, -0.0672,  ..., -0.0170,  0.0236,  0.1210]],

        [[-0.0731, -0.0159, -0.0946,  ..., -0.0348,  0.0121,  0.0863],
         [-0.0640, -0.0051, -0.0628,  ..., -0.0240,  0.0176,  0.1001],
         [-0.0483, -0.0169, -0.0607,  ..., -0.0187,  0.0228,  0.1026],
         ...,
         [-0.0774, -0.0200, -0.0869,  ..., -0.0214,  0.0187,  0.0963],
         [-0.0674, -0.0212, -0.0852,  ..., -0.0208,  0.0106,  0.0858],
         [-0.0674, -0.0208, -0.0894,  ..., -0.0119,  0.0115,  0.1066]],

        [[-0.0684, -0.0083, -0.0922,  ..., -0.0509,  0.0271,  0.0885],
         [-0.0600, -0.0082, -0.0755,  ..., -0

## Inferencia por búsqueda codiciosa
Creamos una función para generar el idioma objetivo dado el idioma fuente usando el modelo secuencia a secuencia definido anteriormente. Se realiza mediante búsqueda codiciosa (_greedy search_), esto es, se toma la palabra más probable de la salida del decodificador como la siguiente palabra de entrada. Este procedimiento se repite hasta que se tenga el máximo tamaño de secuencia o que la siguiente palabra sea el _token_ especial `<SEP>`.

In [19]:
def traduce_greedy(modelo, src, vocab, t_max = 100):
  # número de oraciones de entrada en el idioma fuente
  n_inp_sent = src.shape[-1]

  # pasamos el texto en el idioma fuente por el codificador
  output, (h, c) = modelo.enc(src)

  # creamos un tensor de tamaño igual al número de oraciones de entrada en el
  # idioma fuente con todos los valores igual al token de fin <SEP>
  idxs = th.full((t_max, n_inp_sent),
                 vocab.str2id_en['<SEP>'],
                 device = src.device,
                 dtype = src.dtype)

  # ponemos valores del primer paso igual al índice del token de inicio <CLS>
  idxs[0] = vocab.str2id_en['<CLS>']

  # # creamos lista de listas vacías para las palabras de cada oración
  # texts = [[] for _ in range(n_inp_sent)]
  for step in range(1, t_max):
    # pasamos lote con últimas palabras y estado anterior del decodificador
    # (último estado del codificador en la primera llamada) para obtener las
    # probabilidades de siguiente palabra
    logits, (h, c) = modelo.dec(idxs[step - 1:step], h, c)

    # tomamos la palabra más probable como siguiente en cada ejemplo del lote
    idxs[step] = logits.argmax(axis = 1)

    # generación termina si todas las siguientes palabras corresponden a <SEP>
    if th.all(idxs[step] == vocab.str2id_en['<SEP>']):
      break

  npidxs = idxs[1:step].permute(1, 0).cpu().numpy()
  texts = [' '.join([vocab.id2str_en[j] for j in s]) for s in npidxs]

  return texts

Traducimos un lote con el modelo sin entrenar.

In [20]:
traduce_greedy(modelo, src_batch, vocab)

['mugs mugs mugs mugs mugs twos mugs mugs mugs mugs mugs mugs mugs mugs mugs mugs mugs mugs mugs lamps mugs mugs mugs mugs mugs mugs mugs mugs mugs mugs mugs Pop mugs mugs typing cakes sitting typing Accompanied typing mugs typing mugs mugs mugs mugs mugs mugs mugs mugs mugs loudspeaker loudspeaker Accompanied Accompanied loudspeaker Accompanied lamps crates stunt loudspeaker acting gut acting lamps lamps lamps mugs mugs mugs mugs mugs states mugs typing mugs Accompanied mugs mugs mugs levels mugs mugs mugs mugs lamps lamps mugs mugs mugs mugs mugs mugs mugs mugs mugs twos mugs',
 'mugs lamps lamps mugs mugs mugs mugs mugs blast mugs mugs mugs refuge mugs mugs mugs mugs mugs mugs mugs loudspeaker loudspeaker loudspeaker loudspeaker casual parts levels mugs mugs mugs mugs mugs mugs mugs parts typing loudspeaker loudspeaker pockets lamps mugs levels mugs mugs mugs mugs mugs mugs mugs mugs blast Armenian discovers casual typing mugs typing typing fashioned fashioned containers lamps typin

## Funciones para el entrenamiento
Para facilitar el entrenamiento, definimos diversas funciones.

### Guardar _checkpoint_
Podemos guardar el estado del entrenamiento y de un modelo mediante la función `save`.

In [21]:
def guarda_ckpt(ckptpath, modelo, epoca, opt):
  estado_modelo = {'epoch': epoca,
                   'model_state_dict': modelo.state_dict(),
                   'optimizer_state_dict': opt.state_dict()}
  th.save(estado_modelo, ckptpath)

### Registrar información para Tensorboard
Para guardar información que pueda visualizarse con Tensorboard, instanciamos la clase `SummaryWriter` del submódulo `tensorboard` del módulo `utils` y escribimos información mediante los distintos métodos disponibles (por ej. `add_scalar`, `add_image`, `add_histogram`, etc.).

In [22]:
def registra_info_tboard(writer, epoca, hist):
  for (m,v) in hist.items():
    writer.add_scalar(m, v[epoca], epoca)


### Paso de entrenamiento
Definimos una función que realiza un paso de entrenamiento, a la cual llamaremos `paso_ent`. Esta función recibe como argumentos:
- El modelo como una instancia de `Module` (`modelo`).
- La función de pérdida (`fp`).
- La métrica a evaluar (`metrica`)
- Una instancia de `Optimizer` del módulo `optim` de PyTorch para actualizar los pesos y sesgos (`opt`).
- Una instancia de `Tensor` con el lote de entradas (`X`).
- Una instancia de `Tensor` con el lote de salidas (`y`).

La función obtiene las predicciones del modelo para las entradas, calcula la pérdida, obtiene los gradientes y actualiza todos los parámetros del modelo mediante el optimizador pasado como argumento.

In [23]:
def paso_ent(modelo,
             fp,
             metrica,
             opt,
             src,
             tgt_inp,
             tgt_out):
  opt.zero_grad() # se ponen los gradientes asociados a los parámetros
                  # a actualizaren en cero

  y_hat = modelo(src, tgt_inp) # se propagan las entradas para obtener las predicciones

  y_hat = y_hat.reshape(-1, y_hat.shape[-1])
  tgt_out = tgt_out.permute(1, 0).reshape(-1)

  perdida = fp(y_hat, tgt_out) # se calcula la pérdida
  perdida.backward() # se obtienen los gradientes
  nn.utils.clip_grad_norm_(modelo.parameters(),
                           max_norm = 1.0) # limita norma de gradientes
  opt.step() # se actualizan todos los parámetros del modelo

  with th.no_grad():
    perdida_paso = perdida.cpu().numpy() # convertimos la pérdida (instancia de
                                         # Tensor de orden 0) a NumPy, para
                                         # lo que es necesario moverla a CPU
    metricas_paso = metrica(y_hat, tgt_out)

  return perdida_paso, metricas_paso

### Ciclo principal
También definimos el ciclo de entrenamiento principal, que llamaremos `entrena`. Esta función recibe como argumentos:
- El modelo como una instancia de `Module` (`modelo`).
- La función de pérdida (`fp`).
- La métrica a evaluar (`metrica`)
- Una instancia de `Optimizer` del módulo `optim` de PyTorch para actualizar los pesos y sesgos (`opt`).
- El cargador de datos del subconjunto de entrenamiento (`entdl`).
- El cargador de datos del subconjunto de validación (`valdl`).
- El dispositivo en el que se ejecutará el modelo (`disp`).
- La ruta al archivo donde se guardará el estado del entrenamiento y el modelo (`ckptpath`).
- El número de épocas de entrenamiento (`n_epocas`).
- La ruta al directorio donde se guardará la información para Tensorboard (`tbdir`).

Para cada una de las épocas, va tomando un lote de entradas y salidas a la vez, con los cuales se realiza un paso de entrenamiento.

In [24]:
def entrena(modelo,
            fp,
            metrica,
            opt,
            entdl,
            valdl,
            disp,
            ckptpath,
            n_epocas = 10,
            tbdir = 'runs/'):
  n_lotes_ent = len(entdl)
  n_lotes_val = len(valdl)

  hist = {
    'ent': {
    'perdida': np.zeros(n_epocas),
    metrica.__name__: np.zeros(n_epocas)
    },
    'val': {
    'perdida': np.zeros(n_epocas),
    metrica.__name__: np.zeros(n_epocas)
    }
  }

  tbwriter_ent = SummaryWriter(tbdir + '/ent')
  tbwriter_val = SummaryWriter(tbdir + '/val')

  perdida_min = th.inf
  mejor_modelo = copy.deepcopy(modelo)

  for e in range(n_epocas):
    # bucle de entrenamiento
    modelo.train()
    for src,tgt_inp,tgt_out in entdl:
      src = src.to(disp)
      tgt_inp = tgt_inp.to(disp)
      tgt_out = tgt_out.to(disp)

      perdida_paso, metrica_paso = paso_ent(modelo,
                                            fp,
                                            metrica,
                                            opt,
                                            src,
                                            tgt_inp,
                                            tgt_out)

      hist["ent"]['perdida'][e] += perdida_paso
      hist["ent"][metrica.__name__][e] += metrica_paso

    # bucle de validación
    modelo.eval()
    with th.no_grad():
      for src,tgt_inp,tgt_out in valdl:
        src = src.to(disp)
        tgt_inp = tgt_inp.to(disp)
        tgt_out = tgt_out.to(disp)

        y_hat = modelo(src, tgt_inp)

        tgt_out = tgt_out.permute(1, 0).reshape(-1)
        y_hat = y_hat.reshape(-1, y_hat.shape[-1])

        hist['val']['perdida'][e] += fp(y_hat, tgt_out)
        hist['val'][metrica.__name__][e] += metrica(y_hat, tgt_out)

    hist['ent']['perdida'][e] /=  n_lotes_ent
    hist['ent'][metrica.__name__][e] /= n_lotes_ent
    hist['val']['perdida'][e] /=  n_lotes_val
    hist['val'][metrica.__name__][e] /= n_lotes_val

    # guardamos checkpoint y copiamos pesos y sesgos del modelo
    # actual si disminuye la metrica a monitorear
    if hist['val']['perdida'][e] < perdida_min:
      mejor_modelo.load_state_dict(modelo.state_dict())
      # guarda_ckpt(ckptpath, modelo, e, opt)
      perdida_min = hist['val']['perdida'][e]

    registra_info_tboard(tbwriter_ent, e, hist['ent'])
    registra_info_tboard(tbwriter_val, e, hist['val'])

    print(f'Época {e}: '
          f'Perdida(E) = {hist["ent"]["perdida"][e]:.3f}, '
          f'{metrica.__name__}(E) = {hist["ent"][metrica.__name__][e]:.3f}, '
          f'Perdida(V) = {hist["val"]["perdida"][e]:.3f}, '
          f'{metrica.__name__}(V) = {hist["val"][metrica.__name__][e]:.3f}')

    if e % 5 == 0:
      print(f'\tSRC: {" ".join(convert_indexes(src[1:-1, 0:1], modelo.vocab.id2str_fr))}\n'
            f'\tTGT: {traduce_greedy(modelo, src[:, 0:1], modelo.vocab)[0]}')

  return modelo, mejor_modelo, hist

## Entrenamiento y evaluación
Instanciamos el criterio para la pérdida, el optimizador y llamamos a la función `entrena`.

In [25]:
fp = nn.CrossEntropyLoss()
opt = th.optim.AdamW(modelo.parameters(),
                     lr = 1e-3,
                     weight_decay = 1e-3)
modelo, mejor_modelo, hist = entrena(modelo,
                                     fp,
                                     nn.functional.cross_entropy,
                                     opt,
                                     entdl,
                                     valdl,
                                     disp = DEVICE,
                                     ckptpath = './',
                                     n_epocas = N_EPOCAS,
                                     tbdir = './')

Época 0: Perdida(E) = 2.939, cross_entropy(E) = 2.939, Perdida(V) = 2.501, cross_entropy(V) = 2.501
	SRC: Deux hommes de deux équipes adverses courent vers un ballon de football . <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS>
	TGT: A man in a and and and and in a .
Época 1: Perdida(E) = 2.341, cross_entropy(E) = 2.341, Perdida(V) = 2.221, cross_entropy(V) = 2.221
Época 2: Perdida(E) = 2.106, cross_entropy(E) = 2.106, Perdida(V) = 2.060, cross_entropy(V) = 2.060
Época 3: Perdida(E) = 1.969, cross_entropy(E) = 1.969, Perdida(V) = 1.950, cross_entropy(V) = 1.950
Época 4: Perdida(E) = 1.873, cross_entropy(E) = 1.873, Perdida(V) = 1.861, cross_entropy(V) = 1.861
Época 5: Perdida(E) = 1.766, cross_entropy(E) = 1.766, Perdida(V) = 1.793, cross_entropy(V) = 1.793
	SRC: Deux hommes de deux équipes adverses courent vers un ballon de football . <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS> <CLS>
	TGT: Two men are playing a man , a man .
Ép

Traducimos un lote con el modelo entrenado.

In [26]:
traduce_greedy(modelo, src_batch, vocab)

['A brunette woman is doing a trick on the sidewalk . <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A man is riding a trick in a race <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A crowd of people are gathered with two children are sitting on the ground , in a room . <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A street performer on the floor is at a rodeo at a rodeo . <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A woman is showing her bags . <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A man wearing a black shirt is working on a roof . <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A dog is running over a red slide . <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP> <SEP>',
 'A young boy i